In [ ]:
!pip -q install -U huggingface_hub requests tqdm

In [ ]:
import os
import json
import time
import hashlib
import shutil
from pathlib import Path

import requests
from tqdm.auto import tqdm

from huggingface_hub import HfApi, hf_hub_download, login
# ============================================================
# PROJECT CONFIGURATION
# ============================================================

HF_REPO_ID = "ndeda/neural-object-reconstruction-hdris"
HF_REPO_TYPE = "dataset"

# Temporary Colab workspace
WORK_DIR = Path("/content/neural_object_hdris")
DOWNLOAD_DIR = WORK_DIR / "downloads"
METADATA_DIR = WORK_DIR / "metadata"
STATE_FILE = WORK_DIR / "download_state.json"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

print("Workspace:", WORK_DIR)
print("HDRI download directory:", DOWNLOAD_DIR)
print("State file:", STATE_FILE)

Workspace: /content/neural_object_hdris
HDRI download directory: /content/neural_object_hdris/downloads
State file: /content/neural_object_hdris/download_state.json


In [ ]:
from huggingface_hub import notebook_login

notebook_login()
api = HfApi()

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    exist_ok=True,
)

print(f"Repository ready: https://huggingface.co/datasets/{HF_REPO_ID}")

Repository ready: https://huggingface.co/datasets/ndeda/neural-object-reconstruction-hdris


In [ ]:
def load_state():
    if STATE_FILE.exists():
        with open(STATE_FILE, "r", encoding="utf-8") as f:
            return json.load(f)

    return {
        "version": 1,
        "assets": {}
    }


state = load_state()

print(f"Tracked HDRIs: {len(state['assets'])}")

Tracked HDRIs: 0


In [ ]:
def save_state():
    temp_file = STATE_FILE.with_suffix(".tmp")

    with open(temp_file, "w", encoding="utf-8") as f:
        json.dump(state, f, indent=2, ensure_ascii=False)

    temp_file.replace(STATE_FILE)

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    sha256 = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()

In [ ]:
files = list(api.list_repo_files(
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
))

print("Files currently in repository:")

if files:
    for file in files:
        print(" -", file)
else:
    print("Repository is currently empty.")

Files currently in repository:
 - .gitattributes


In [ ]:
# ============================================================
# CELL 8 — RESUMABLE POLY HAVEN HDRI DOWNLOADER
# ============================================================

import requests
import hashlib
import json
from pathlib import Path
from tqdm.auto import tqdm

POLYHAVEN_API = "https://api.polyhaven.com"

# Use a unique User-Agent as requested by Poly Haven.
POLYHAVEN_HEADERS = {
    "User-Agent": "NeuralObjectReconstructionLab/1.0 (academic research)"
}

# We deliberately use 2K HDRIs.
HDRI_RESOLUTION = "2k"
HDRI_FORMAT = "hdr"


def get_polyhaven_asset_files(asset_id):
    """
    Get the available files and download URLs for one Poly Haven asset.
    """
    url = f"{POLYHAVEN_API}/files/{asset_id}"

    response = requests.get(
        url,
        headers=POLYHAVEN_HEADERS,
        timeout=60
    )

    response.raise_for_status()
    return response.json()


def get_hdri_2k_info(asset_id):
    """
    Return the 2K HDR file information for a Poly Haven HDRI.
    """

    files = get_polyhaven_asset_files(asset_id)

    try:
        info = files["hdri"][HDRI_RESOLUTION][HDRI_FORMAT]
    except KeyError:
        raise RuntimeError(
            f"2K HDR file not available for Poly Haven asset: {asset_id}"
        )

    return {
        "url": info["url"],
        "size": info["size"],
        "md5": info["md5"],
    }


def md5_file(path, chunk_size=1024 * 1024):
    """
    Calculate MD5 checksum without loading the entire HDRI into RAM.
    """

    md5 = hashlib.md5()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            md5.update(chunk)

    return md5.hexdigest()


def verify_md5(path, expected_md5):
    """
    Verify a downloaded file against Poly Haven's official MD5.
    """

    actual_md5 = md5_file(path)

    return actual_md5.lower() == expected_md5.lower()


def download_resumable(url, destination, expected_size=None):
    """
    Download a file with HTTP resume support.

    If a partial file already exists, continue from where it stopped.
    """

    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    existing_size = destination.stat().st_size if destination.exists() else 0

    headers = dict(POLYHAVEN_HEADERS)

    if existing_size > 0:
        headers["Range"] = f"bytes={existing_size}-"

    response = requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=120
    )

    # Server supports our requested range.
    if existing_size > 0 and response.status_code == 206:

        mode = "ab"

        total_remaining = int(
            response.headers.get("Content-Length", 0)
        )

        total_size = existing_size + total_remaining

        print(
            f"Resuming download: "
            f"{existing_size / 1024**2:.2f} MB already present"
        )

    # Server ignored Range; start again safely.
    elif existing_size > 0 and response.status_code == 200:

        print(
            "Server did not provide a resumable response. "
            "Restarting this individual download."
        )

        mode = "wb"

        total_size = int(
            response.headers.get("Content-Length", 0)
        )

    elif response.status_code == 200:

        mode = "wb"

        total_size = int(
            response.headers.get("Content-Length", 0)
        )

    else:
        response.raise_for_status()

    with open(destination, mode) as f:

        with tqdm(
            total=total_size if total_size else None,
            initial=existing_size if mode == "ab" else 0,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=destination.name
        ) as progress:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if not chunk:
                    continue

                f.write(chunk)
                progress.update(len(chunk))

    # Final size check when expected size is known.
    if expected_size is not None:

        actual_size = destination.stat().st_size

        if actual_size != expected_size:

            raise RuntimeError(
                f"Size mismatch for {destination.name}: "
                f"expected {expected_size:,} bytes, "
                f"got {actual_size:,} bytes."
            )

    return destination

In [ ]:
def download_hdri(asset_id):
    """
    Download one Poly Haven HDRI safely and resumably.

    Already verified HDRIs are skipped.
    """

    asset_id = str(asset_id)

    # --------------------------------------------------------
    # Check persistent state first
    # --------------------------------------------------------

    asset_state = state["assets"].get(asset_id, {})

    if (
        asset_state.get("downloaded") is True
        and asset_state.get("verified") is True
    ):

        local_path = DOWNLOAD_DIR / asset_state["filename"]

        if local_path.exists():

            print(f"[SKIP] {asset_id} already verified.")

            return local_path

    # --------------------------------------------------------
    # Ask Poly Haven for the official download information
    # --------------------------------------------------------

    info = get_hdri_2k_info(asset_id)

    filename = f"{asset_id}_{HDRI_RESOLUTION}.{HDRI_FORMAT}"

    destination = DOWNLOAD_DIR / filename

    print(f"\nProcessing: {asset_id}")
    print(f"Resolution: {HDRI_RESOLUTION}")
    print(f"Expected size: {info['size'] / 1024**2:.2f} MB")

    # --------------------------------------------------------
    # If an existing file has the correct size, verify it
    # before downloading anything.
    # --------------------------------------------------------

    if destination.exists():

        current_size = destination.stat().st_size

        if current_size == info["size"]:

            print("File already has the expected size.")
            print("Verifying checksum...")

            if verify_md5(destination, info["md5"]):

                print("[OK] Existing file verified.")

                state["assets"][asset_id] = {
                    "filename": filename,
                    "downloaded": True,
                    "verified": True,
                    "size": info["size"],
                    "md5": info["md5"],
                    "source": "Poly Haven",
                    "resolution": HDRI_RESOLUTION,
                    "format": HDRI_FORMAT,
                }

                save_state()

                return destination

            else:

                print(
                    "[WARNING] Checksum mismatch. "
                    "Removing corrupted file."
                )

                destination.unlink()

    # --------------------------------------------------------
    # Download / resume
    # --------------------------------------------------------

    download_resumable(
        url=info["url"],
        destination=destination,
        expected_size=info["size"]
    )

    # --------------------------------------------------------
    # Verify
    # --------------------------------------------------------

    print("Checking MD5 checksum...")

    if not verify_md5(destination, info["md5"]):

        print("[ERROR] Checksum verification failed.")

        destination.unlink(missing_ok=True)

        raise RuntimeError(
            f"Checksum verification failed for {asset_id}"
        )

    print("[OK] Download verified.")

    # --------------------------------------------------------
    # Save persistent state
    # --------------------------------------------------------

    state["assets"][asset_id] = {
        "filename": filename,
        "downloaded": True,
        "verified": True,
        "size": info["size"],
        "md5": info["md5"],
        "source": "Poly Haven",
        "resolution": HDRI_RESOLUTION,
        "format": HDRI_FORMAT,
    }

    save_state()

    print(f"[DONE] {asset_id}")

    return destination

In [ ]:
# ============================================================
# CELL 9 — GET POLY HAVEN HDRI CATALOGUE
# ============================================================

POLYHAVEN_ASSETS_URL = f"{POLYHAVEN_API}/assets"

response = requests.get(
    POLYHAVEN_ASSETS_URL,
    headers=POLYHAVEN_HEADERS,
    timeout=120
)

response.raise_for_status()

all_assets = response.json()

print(f"Total Poly Haven assets found: {len(all_assets):,}")

# Poly Haven uses type=0 for HDRIs.
hdri_catalogue = {
    asset_id: metadata
    for asset_id, metadata in all_assets.items()
    if metadata.get("type") == 0
}

print(f"HDRIs found: {len(hdri_catalogue):,}")

Total Poly Haven assets found: 2,355
HDRIs found: 986


In [ ]:
# ============================================================
# CELL 10 — SAVE HDRI CATALOGUE
# ============================================================

CATALOGUE_FILE = METADATA_DIR / "polyhaven_hdri_catalogue.json"

with open(CATALOGUE_FILE, "w", encoding="utf-8") as f:
    json.dump(
        hdri_catalogue,
        f,
        indent=2,
        ensure_ascii=False
    )

print(f"Saved catalogue to:")
print(CATALOGUE_FILE)

Saved catalogue to:
/content/neural_object_hdris/metadata/polyhaven_hdri_catalogue.json


In [ ]:
# ============================================================
# CELL 11 — INSPECT HDRI CATEGORIES
# ============================================================

from collections import Counter

category_counts = Counter()

for asset_id, metadata in hdri_catalogue.items():
    category = metadata.get("category", "Unknown")
    category_counts[category] += 1

print("HDRI categories:\n")

for category, count in category_counts.most_common():
    print(f"{count:5d}  {category}")

HDRI categories:

   74  Fields & Countryside/Grassland & Meadows/Open Fields & Meadows
   58  Studio/Photo Studios/Softbox & Lamp Setups
   51  Mountains & Hills/Hilltops & Viewpoints/Grassland Hilltops
   33  Mountains & Hills/Hilltops & Viewpoints/Lakeside Hilltops
   30  Mountains & Hills/Hilltops & Viewpoints/Rocky Summits
   23  Abandoned & Ruins/Industrial/Derelict Factories & Warehouses
   23  Parks & Gardens/Parks/Open Parkland
   20  Streets & Town/Squares & Plazas/Historic & European Squares
   17  Interiors/Workshops & Garages/Industrial Workshops
   17  Parks & Gardens/Parks/Tree-lined Paths
   16  Coast & Water/Rivers & Lakes/Lakeshores
   14  Forest & Woodland/Woodland Paths/Forest Trails
   14  Streets & Town/Squares & Plazas/Courtyards & Quads
   14  Studio/Photo Studios/Coloured & Gel Lighting
   13  Abandoned & Ruins/Buildings/Derelict Interiors
   13  Studio/Photo Studios/Themed & Set-Dressed
   12  Coast & Water/Rivers & Lakes/Riverbanks
   12  Studio/Photo Studios

In [ ]:
# ============================================================
# CELL 12 — INSPECT HDRI LIGHTING ATTRIBUTES
# ============================================================

attribute_values = {
    "time_of_day": Counter(),
    "weather": Counter(),
    "light_type": Counter(),
    "contrast": Counter(),
    "environment": Counter(),
    "sky_view": Counter(),
}

for asset_id, metadata in hdri_catalogue.items():

    attributes = metadata.get("attributes", {})

    for key in attribute_values:

        value = attributes.get(key)

        if value is not None:
            attribute_values[key][str(value)] += 1


for attribute_name, values in attribute_values.items():

    print("\n" + "=" * 60)
    print(attribute_name)

    for value, count in values.most_common():
        print(f"{count:5d}  {value}")


time_of_day
  170  midday
  169  afternoon
  132  sunset
  112  morning
   64  sunrise
   46  night
   19  dusk

weather
  317  clear
  316  partly_cloudy
  172  overcast
   11  fog
    3  rain
    1  snow

light_type
  750  natural
  148  artificial
   88  mixed

contrast
  363  low
  354  high
  269  medium

environment
  709  outdoor
  277  indoor

sky_view
  502  open
  207  obstructed


In [ ]:
# ============================================================
# CELL 13 — FIND HDRIs WITH 2K HDR FILES
# ============================================================

def get_2k_hdr_info(asset_id):

    try:
        files = get_polyhaven_asset_files(asset_id)

        return files["hdri"]["2k"]["hdr"]

    except (KeyError, requests.RequestException):
        return None


hdri_2k_available = {}

print("Checking 2K HDR availability...")

for i, asset_id in enumerate(hdri_catalogue, start=1):

    info = get_2k_hdr_info(asset_id)

    if info is not None:
        hdri_2k_available[asset_id] = info

    if i % 100 == 0:
        print(
            f"Checked {i:,}/{len(hdri_catalogue):,}"
        )

print(
    f"\nHDRIs with 2K HDR available: "
    f"{len(hdri_2k_available):,}"
)

Checking 2K HDR availability...
Checked 100/986
Checked 200/986
Checked 300/986
Checked 400/986
Checked 500/986
Checked 600/986
Checked 700/986
Checked 800/986
Checked 900/986

HDRIs with 2K HDR available: 986


In [24]:
# ============================================================
# CELL 14 — STORAGE ESTIMATE
# ============================================================

total_2k_bytes = sum(
    info["size"]
    for info in hdri_2k_available.values()
)

print(
    f"Total available 2K HDR storage: "
    f"{total_2k_bytes / 1024**3:.2f} GB"
)

print(
    f"Average 2K HDR size: "
    f"{total_2k_bytes / len(hdri_2k_available) / 1024**2:.2f} MB"
)

Total available 2K HDR storage: 5.86 GB
Average 2K HDR size: 6.08 MB


In [25]:
# ============================================================
# CELL 15 — CREATE DIVERSE HDRI CANDIDATES
# ============================================================

def get_attr(metadata, key):
    return metadata.get("attributes", {}).get(key)


candidate_groups = {
    "daylight": [],
    "overcast": [],
    "sunrise_sunset": [],
    "indoor": [],
    "studio": [],
    "urban": [],
    "nature": [],
    "high_contrast": [],
    "low_contrast": [],
}

for asset_id, metadata in hdri_catalogue.items():

    if asset_id not in hdri_2k_available:
        continue

    category = metadata.get("category", "").lower()
    tags = " ".join(
        str(x).lower()
        for x in metadata.get("tags", [])
    )

    combined = f"{category} {tags}"

    environment = str(
        get_attr(metadata, "environment")
    ).lower()

    weather = str(
        get_attr(metadata, "weather")
    ).lower()

    contrast = str(
        get_attr(metadata, "contrast")
    ).lower()

    time_of_day = str(
        get_attr(metadata, "time_of_day")
    ).lower()

    if (
        "indoor" in combined
        or environment == "indoor"
        or "interior" in combined
    ):
        candidate_groups["indoor"].append(asset_id)

    if (
        "studio" in combined
        or "studio" in tags
    ):
        candidate_groups["studio"].append(asset_id)

    if (
        environment == "outdoor"
        and (
            "day" in time_of_day
            or "sun" in combined
        )
    ):
        candidate_groups["daylight"].append(asset_id)

    if (
        "overcast" in weather
        or "cloud" in weather
        or "overcast" in combined
    ):
        candidate_groups["overcast"].append(asset_id)

    if (
        "sunset" in combined
        or "sunrise" in combined
        or time_of_day in {"dawn", "dusk", "sunrise", "sunset"}
    ):
        candidate_groups["sunrise_sunset"].append(asset_id)

    if (
        "city" in combined
        or "urban" in combined
        or "street" in combined
        or "building" in combined
    ):
        candidate_groups["urban"].append(asset_id)

    if (
        "nature" in combined
        or "forest" in combined
        or "mountain" in combined
        or "park" in combined
        or "field" in combined
    ):
        candidate_groups["nature"].append(asset_id)

    if contrast == "high":
        candidate_groups["high_contrast"].append(asset_id)

    if contrast == "low":
        candidate_groups["low_contrast"].append(asset_id)


for group, assets in candidate_groups.items():

    print(
        f"{group:20s}: "
        f"{len(assets):5d} candidates"
    )

daylight            :   365 candidates
overcast            :   488 candidates
sunrise_sunset      :   215 candidates
indoor              :   279 candidates
studio              :   103 candidates
urban               :   231 candidates
nature              :   470 candidates
high_contrast       :   354 candidates
low_contrast        :   363 candidates


In [26]:
# ============================================================
# CELL 16 — DISPLAY CANDIDATE HDRIs
# ============================================================

for group, asset_ids in candidate_groups.items():

    print("\n")
    print("=" * 80)
    print(group.upper())
    print("=" * 80)

    for asset_id in asset_ids[:30]:

        metadata = hdri_catalogue[asset_id]

        print(
            f"{asset_id:35s} | "
            f"{metadata.get('name', '')}"
        )



DAYLIGHT
aarfontein_dirt_road                | Aarfontein Dirt Road
abandoned_hopper_terminal_01        | Abandoned Hopper Terminal 01
abandoned_hopper_terminal_02        | Abandoned Hopper Terminal 02
abandoned_hopper_terminal_03        | Abandoned Hopper Terminal 03
abandoned_hopper_terminal_04        | Abandoned Hopper Terminal 04
abandoned_parking                   | Abandoned Parking
abandoned_pathway                   | Abandoned Pathway
abandoned_slipway                   | Abandoned Slipway
abandoned_tank_farm_01              | Abandoned Tank Farm 01
abandoned_tank_farm_02              | Abandoned Tank Farm 02
abandoned_tank_farm_03              | Abandoned Tank Farm 03
abandoned_tank_farm_04              | Abandoned Tank Farm 04
abandoned_tank_farm_05              | Abandoned Tank Farm 05
adams_place_bridge                  | Adams Place Bridge
ahornsteig                          | Ahornsteig
air_museum_playground               | Air Museum Playground
aloe_farm_shade_house  

In [28]:
# ============================================================
# CELL 17 — AUTOMATICALLY SELECT 24 DIVERSE HDRIs
# ============================================================

import random

TARGET_HDRIS = 24
SELECTION_SEED = 20260814

random.seed(SELECTION_SEED)

# Make sure we only consider assets with 2K HDR files.
available_ids = set(hdri_2k_available.keys())

# ------------------------------------------------------------
# Helper functions
# ------------------------------------------------------------

def asset_text(asset_id):
    metadata = hdri_catalogue[asset_id]

    parts = [
        str(metadata.get("name", "")),
        str(metadata.get("category", "")),
        " ".join(
            str(x) for x in metadata.get("tags", [])
        ),
    ]

    attributes = metadata.get("attributes", {})

    for key, value in attributes.items():
        parts.append(str(key))
        parts.append(str(value))

    return " ".join(parts).lower()


# ------------------------------------------------------------
# Candidate groups
# ------------------------------------------------------------

groups = {
    "studio": [],
    "indoor": [],
    "overcast": [],
    "sunrise_sunset": [],
    "urban": [],
    "nature": [],
    "daylight": [],
    "high_contrast": [],
    "low_contrast": [],
}

for asset_id in available_ids:

    text = asset_text(asset_id)

    metadata = hdri_catalogue[asset_id]
    attributes = metadata.get("attributes", {})

    category = str(
        metadata.get("category", "")
    ).lower()

    contrast = str(
        attributes.get("contrast", "")
    ).lower()

    time_of_day = str(
        attributes.get("time_of_day", "")
    ).lower()

    weather = str(
        attributes.get("weather", "")
    ).lower()

    # Studio
    if "studio" in text:
        groups["studio"].append(asset_id)

    # Indoor
    if (
        "indoor" in text
        or "interior" in text
        or "room" in text
    ):
        groups["indoor"].append(asset_id)

    # Overcast
    if (
        "overcast" in text
        or "cloudy" in text
        or "cloud" in weather
    ):
        groups["overcast"].append(asset_id)

    # Sunrise / sunset
    if (
        "sunrise" in text
        or "sunset" in text
        or time_of_day in {
            "sunrise",
            "sunset",
            "dawn",
            "dusk",
        }
    ):
        groups["sunrise_sunset"].append(asset_id)

    # Urban
    if any(
        word in text
        for word in [
            "urban",
            "city",
            "street",
            "building",
            "parking",
            "road",
        ]
    ):
        groups["urban"].append(asset_id)

    # Nature
    if any(
        word in text
        for word in [
            "nature",
            "forest",
            "mountain",
            "field",
            "grass",
            "park",
            "beach",
            "coast",
        ]
    ):
        groups["nature"].append(asset_id)

    # Daylight
    if (
        "daylight" in text
        or "midday" in text
        or time_of_day in {
            "day",
            "midday",
            "noon",
        }
    ):
        groups["daylight"].append(asset_id)

    # Contrast
    if contrast == "high":
        groups["high_contrast"].append(asset_id)

    if contrast == "low":
        groups["low_contrast"].append(asset_id)


# ------------------------------------------------------------
# Select balanced representatives
# ------------------------------------------------------------

selected = []
selected_set = set()

# We want a broad distribution rather than 24 similar HDRIs.
quotas = {
    "studio": 3,
    "indoor": 3,
    "overcast": 3,
    "sunrise_sunset": 3,
    "urban": 3,
    "nature": 3,
    "daylight": 3,
    "high_contrast": 2,
    "low_contrast": 1,
}

for group, quota in quotas.items():

    candidates = [
        x for x in groups[group]
        if x not in selected_set
    ]

    random.shuffle(candidates)

    for asset_id in candidates[:quota]:

        selected.append(asset_id)
        selected_set.add(asset_id)


# ------------------------------------------------------------
# Fill remaining slots if some groups had insufficient assets.
# ------------------------------------------------------------

remaining = [
    x for x in available_ids
    if x not in selected_set
]

random.shuffle(remaining)

while len(selected) < TARGET_HDRIS and remaining:

    asset_id = remaining.pop()

    selected.append(asset_id)
    selected_set.add(asset_id)


SELECTED_HDRIS = selected[:TARGET_HDRIS]

print("=" * 70)
print(f"SELECTED HDRIs: {len(SELECTED_HDRIS)}")
print("=" * 70)

for i, asset_id in enumerate(SELECTED_HDRIS, 1):

    metadata = hdri_catalogue[asset_id]

    print(
        f"{i:02d}. "
        f"{asset_id:30s} | "
        f"{metadata.get('name', '')}"
    )

if len(SELECTED_HDRIS) < TARGET_HDRIS:
    print(
        f"\nWARNING: only "
        f"{len(SELECTED_HDRIS)} HDRIs could be selected."
    )

SELECTED HDRIs: 24
01. art_studio                     | Art Studio
02. pav_studio_01                  | PAV Studio 01
03. wooden_studio_08               | Wooden Studio 08
04. wooden_studio_10               | Wooden Studio 10
05. monochrome_studio_02           | Monochrome Studio 02
06. brown_photostudio_01           | Brown Photostudio 01
07. turning_area                   | Turning Area
08. teufelsberg_ground_1           | Teufelsberg Ground 1
09. rural_evening_road             | Rural Evening Road
10. spruit_sunrise                 | Spruit Sunrise
11. ninomaru_teien                 | Ninomaru Teien
12. bell_park_dawn                 | Bell Park Dawn
13. san_giuseppe_bridge            | San Giuseppe Bridge
14. summer_stage_02                | Summer Stage 02
15. zwartkops_curve_morning        | Zwartkops Curve Morning
16. scythian_tombs                 | Scythian Tombs
17. golden_bay                     | Golden Bay
18. syferfontein_0d_clear_puresky  | Syferfontein 0d Clear (Pure Sk

In [29]:
# ============================================================
# CELL 18 — VALIDATE SELECTED HDRIs
# ============================================================

assert len(SELECTED_HDRIS) > 0

SELECTED_HDRIS = list(
    dict.fromkeys(SELECTED_HDRIS)
)

for asset_id in SELECTED_HDRIS:

    if asset_id not in hdri_catalogue:
        raise ValueError(
            f"Asset not found in Poly Haven catalogue: {asset_id}"
        )

    if asset_id not in hdri_2k_available:
        raise ValueError(
            f"2K HDR unavailable for: {asset_id}"
        )

print(
    f"[OK] {len(SELECTED_HDRIS)} HDRIs "
    f"passed validation."
)

[OK] 24 HDRIs passed validation.


In [30]:
# ============================================================
# CELL 19 — BUILD SELECTED HDRI METADATA
# ============================================================

selected_metadata = {}

for asset_id in SELECTED_HDRIS:

    source = hdri_catalogue[asset_id]
    file_info = hdri_2k_available[asset_id]

    selected_metadata[asset_id] = {
        "asset_id": asset_id,
        "name": source.get("name"),
        "description": source.get("description"),
        "category": source.get("category"),
        "tags": source.get("tags", []),
        "attributes": source.get("attributes", {}),
        "authors": source.get("authors", {}),
        "date_published": source.get("date_published"),
        "source": "Poly Haven",
        "source_url": f"https://polyhaven.com/a/{asset_id}",
        "license": "CC0",
        "resolution": "2k",
        "format": "hdr",
        "download_url": file_info["url"],
        "size_bytes": file_info["size"],
        "md5": file_info["md5"],
    }

print(
    f"[OK] Metadata prepared for "
    f"{len(selected_metadata)} HDRIs."
)

[OK] Metadata prepared for 24 HDRIs.


In [31]:
# ============================================================
# CELL 20 — SAVE SELECTED METADATA
# ============================================================

SELECTED_METADATA_FILE = (
    METADATA_DIR / "selected_hdris.json"
)

with open(
    SELECTED_METADATA_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        selected_metadata,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    f"[OK] Metadata saved:\n"
    f"{SELECTED_METADATA_FILE}"
)

[OK] Metadata saved:
/content/neural_object_hdris/metadata/selected_hdris.json


In [32]:
# ============================================================
# CELL 21 — RESUMABLE DOWNLOAD FUNCTION
# ============================================================

def download_resumable_v2(
    url,
    destination,
    expected_size=None,
    expected_md5=None,
):

    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    existing_size = (
        destination.stat().st_size
        if destination.exists()
        else 0
    )

    # --------------------------------------------------------
    # Already complete and verified?
    # --------------------------------------------------------

    if (
        expected_size is not None
        and existing_size == expected_size
    ):

        if (
            expected_md5 is not None
            and verify_md5(
                destination,
                expected_md5
            )
        ):

            print(
                f"[SKIP] Already verified: "
                f"{destination.name}"
            )

            return destination

    # --------------------------------------------------------
    # Prepare request
    # --------------------------------------------------------

    headers = dict(
        POLYHAVEN_HEADERS
    )

    if existing_size > 0:
        headers["Range"] = (
            f"bytes={existing_size}-"
        )

    response = requests.get(
        url,
        headers=headers,
        stream=True,
        timeout=180
    )

    # --------------------------------------------------------
    # Resume supported
    # --------------------------------------------------------

    if (
        existing_size > 0
        and response.status_code == 206
    ):

        mode = "ab"

        remaining = int(
            response.headers.get(
                "Content-Length",
                0
            )
        )

        total_size = (
            existing_size + remaining
        )

        progress_start = existing_size

        print(
            f"[RESUME] {destination.name} "
            f"from {existing_size / 1024**2:.2f} MB"
        )

    # --------------------------------------------------------
    # Server doesn't support resume
    # --------------------------------------------------------

    elif response.status_code == 200:

        mode = "wb"

        total_size = int(
            response.headers.get(
                "Content-Length",
                0
            )
        )

        progress_start = 0

        if existing_size > 0:
            print(
                f"[RESTART] Server did not support "
                f"resume for {destination.name}"
            )

    else:

        response.raise_for_status()

    # --------------------------------------------------------
    # Download
    # --------------------------------------------------------

    with open(
        destination,
        mode
    ) as f:

        with tqdm(
            total=total_size or None,
            initial=progress_start,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
            desc=destination.name
        ) as progress:

            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):

                if not chunk:
                    continue

                f.write(chunk)
                progress.update(
                    len(chunk)
                )

    # --------------------------------------------------------
    # Size verification
    # --------------------------------------------------------

    actual_size = destination.stat().st_size

    if (
        expected_size is not None
        and actual_size != expected_size
    ):

        raise RuntimeError(
            f"Size mismatch for "
            f"{destination.name}: "
            f"expected {expected_size}, "
            f"got {actual_size}"
        )

    # --------------------------------------------------------
    # MD5 verification
    # --------------------------------------------------------

    if expected_md5 is not None:

        print(
            f"Verifying {destination.name}..."
        )

        if not verify_md5(
            destination,
            expected_md5
        ):

            destination.unlink(
                missing_ok=True
            )

            raise RuntimeError(
                f"MD5 verification failed: "
                f"{destination.name}"
            )

    print(
        f"[OK] Verified: "
        f"{destination.name}"
    )

    return destination

In [33]:
# ============================================================
# CELL 22 — DOWNLOAD ALL SELECTED HDRIs
# ============================================================

download_results = []

for index, asset_id in enumerate(
    SELECTED_HDRIS,
    start=1
):

    print("\n" + "=" * 70)

    metadata = selected_metadata[asset_id]

    filename = (
        f"{asset_id}_2k.hdr"
    )

    destination = (
        DOWNLOAD_DIR / filename
    )

    print(
        f"[{index}/{len(SELECTED_HDRIS)}] "
        f"{asset_id}"
    )

    try:

        path = download_resumable_v2(
            url=metadata["download_url"],
            destination=destination,
            expected_size=metadata["size_bytes"],
            expected_md5=metadata["md5"],
        )

        state["assets"][asset_id] = {
            "asset_id": asset_id,
            "filename": filename,
            "downloaded": True,
            "verified": True,
            "size": metadata["size_bytes"],
            "md5": metadata["md5"],
            "source": "Poly Haven",
            "source_url": metadata["source_url"],
            "license": "CC0",
            "resolution": "2k",
            "format": "hdr",
        }

        save_state()

        download_results.append({
            "asset_id": asset_id,
            "status": "verified",
            "path": str(path),
        })

    except Exception as e:

        print(
            f"[ERROR] {asset_id}: {e}"
        )

        download_results.append({
            "asset_id": asset_id,
            "status": "failed",
            "error": str(e),
        })

        continue


[1/24] art_studio


art_studio_2k.hdr:   0%|          | 0.00/6.41M [00:00<?, ?B/s]

Verifying art_studio_2k.hdr...
[OK] Verified: art_studio_2k.hdr

[2/24] pav_studio_01


pav_studio_01_2k.hdr:   0%|          | 0.00/5.55M [00:00<?, ?B/s]

Verifying pav_studio_01_2k.hdr...
[OK] Verified: pav_studio_01_2k.hdr

[3/24] wooden_studio_08


...

Verifying cloud_layers_2k.hdr...
[OK] Verified: cloud_layers_2k.hdr

[23/24] autumn_hill_view


autumn_hill_view_2k.hdr:   0%|          | 0.00/6.12M [00:00<?, ?B/s]

Verifying autumn_hill_view_2k.hdr...
[OK] Verified: autumn_hill_view_2k.hdr

[24/24] lebombo


lebombo_2k.hdr:   0%|          | 0.00/5.73M [00:00<?, ?B/s]

Verifying lebombo_2k.hdr...
[OK] Verified: lebombo_2k.hdr


In [34]:
# ============================================================
# CELL 23 — DOWNLOAD STATUS
# ============================================================

verified = [
    x for x in download_results
    if x["status"] == "verified"
]

failed = [
    x for x in download_results
    if x["status"] == "failed"
]

print("=" * 70)
print("DOWNLOAD SUMMARY")
print("=" * 70)

print(f"Verified: {len(verified)}")
print(f"Failed:   {len(failed)}")

if failed:

    print("\nFailed files:")

    for item in failed:

        print(
            f" - {item['asset_id']}: "
            f"{item['error']}"
        )

DOWNLOAD SUMMARY
Verified: 24
Failed:   0


In [35]:
# ============================================================
# CELL 24 — CREATE README
# ============================================================

README_FILE = WORK_DIR / "README.md"

readme_text = """---
pretty_name: Neural Object Reconstruction HDRI Library
tags:
- hdr
- hdri
- synthetic-data
- computer-vision
- 3d-reconstruction
- relighting
- blender
license: cc0-1.0
---

# Neural Object Reconstruction HDRI Library

A curated collection of Poly Haven HDRI environments used
for synthetic Blender rendering for neural object
reconstruction and relighting research.

## Resolution

The initial collection uses 2K HDR files.

This resolution was selected to balance:

- rendering speed
- memory usage
- environment detail
- Colab storage
- synthetic dataset generation throughput

## Source

All HDRIs are sourced from Poly Haven.

https://polyhaven.com/

Poly Haven assets are released under CC0.

## Intended use

The HDRIs are used as controllable illumination environments
for Blender-based synthetic data generation.

The rendering pipeline can vary:

- HDRI selection
- HDRI rotation
- environment strength
- exposure
- additional Blender lights

## Metadata

Each HDRI has associated metadata containing:

- asset ID
- asset name
- source
- source URL
- license
- category
- tags
- lighting attributes
- resolution
- format
- file size
- MD5 checksum

## Reproducibility

HDRI selection and rendering parameters will be recorded
by the synthetic data generation pipeline.
"""

with open(
    README_FILE,
    "w",
    encoding="utf-8"
) as f:

    f.write(readme_text)

print(
    f"[OK] README created: "
    f"{README_FILE}"
)

[OK] README created: /content/neural_object_hdris/README.md


In [36]:
# ============================================================
# CELL 25 — UPLOAD README + METADATA
# ============================================================

# Upload README
api.upload_file(
    path_or_fileobj=str(README_FILE),
    path_in_repo="README.md",
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message="Add HDRI library documentation",
)

# Upload selected metadata
api.upload_file(
    path_or_fileobj=str(SELECTED_METADATA_FILE),
    path_in_repo="metadata/selected_hdris.json",
    repo_id=HF_REPO_ID,
    repo_type=HF_REPO_TYPE,
    commit_message="Add HDRI metadata",
)

print("[OK] README and metadata uploaded.")

[OK] README and metadata uploaded.


In [37]:
# ============================================================
# CELL 26 — UPLOAD VERIFIED HDRIs
# ============================================================

for asset_id in SELECTED_HDRIS:

    asset_state = state["assets"].get(
        asset_id,
        {}
    )

    # Already uploaded in a previous run?
    if asset_state.get("uploaded") is True:

        print(
            f"[SKIP] Already uploaded: "
            f"{asset_id}"
        )

        continue

    metadata = selected_metadata[asset_id]

    filename = (
        f"{asset_id}_2k.hdr"
    )

    local_path = (
        DOWNLOAD_DIR / filename
    )

    if not local_path.exists():

        print(
            f"[ERROR] Local file missing: "
            f"{asset_id}"
        )

        continue

    print(
        f"\nUploading: {asset_id}"
    )

    try:

        api.upload_file(
            path_or_fileobj=str(local_path),
            path_in_repo=f"hdris/{filename}",
            repo_id=HF_REPO_ID,
            repo_type=HF_REPO_TYPE,
            commit_message=(
                f"Add HDRI {asset_id}"
            ),
        )

        state["assets"][asset_id][
            "uploaded"
        ] = True

        save_state()

        print(
            f"[OK] Uploaded: {asset_id}"
        )

    except Exception as e:

        print(
            f"[ERROR] Upload failed "
            f"for {asset_id}: {e}"
        )


Uploading: art_studio


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...wnloads/art_studio_2k.hdr: 100%|##########| 6.72MB / 6.72MB            

[OK] Uploaded: art_studio

Uploading: pav_studio_01


...

[OK] Uploaded: autumn_hill_view

Uploading: lebombo


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../downloads/lebombo_2k.hdr: 100%|##########| 6.01MB / 6.01MB            

[OK] Uploaded: lebombo


In [38]:
# ============================================================
# CELL 27 — VERIFY REMOTE REPOSITORY
# ============================================================

remote_files = list(
    api.list_repo_files(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
    )
)

print(
    f"Remote files found: "
    f"{len(remote_files)}"
)

for filename in remote_files:
    print(" -", filename)

Remote files found: 27
 - .gitattributes
 - README.md
 - hdris/art_studio_2k.hdr
 - hdris/autumn_hill_view_2k.hdr
 - hdris/bell_park_dawn_2k.hdr
 - hdris/brown_photostudio_01_2k.hdr
 - hdris/cloud_layers_2k.hdr
 - hdris/golden_bay_2k.hdr
 - hdris/immenstadter_horn_2k.hdr
 - hdris/kloofendal_43d_clear_2k.hdr
 - hdris/lebombo_2k.hdr
 - hdris/monochrome_studio_02_2k.hdr
 - hdris/ninomaru_teien_2k.hdr
 - hdris/pav_studio_01_2k.hdr
 - hdris/river_alcove_2k.hdr
 - hdris/rural_evening_road_2k.hdr
 - hdris/san_giuseppe_bridge_2k.hdr
 - hdris/scythian_tombs_2k.hdr
 - hdris/spruit_sunrise_2k.hdr
 - hdris/summer_stage_02_2k.hdr
 - hdris/syferfontein_0d_clear_puresky_2k.hdr
 - hdris/teufelsberg_ground_1_2k.hdr
 - hdris/turning_area_2k.hdr
 - hdris/wooden_studio_08_2k.hdr
 - hdris/wooden_studio_10_2k.hdr
 - hdris/zwartkops_curve_morning_2k.hdr
 - metadata/selected_hdris.json


In [39]:
# ============================================================
# CELL 28 — VERIFY EVERY HDRI EXISTS REMOTELY
# ============================================================

missing_remote = []

for asset_id in SELECTED_HDRIS:

    filename = f"{asset_id}_2k.hdr"

    remote_path = f"hdris/{filename}"

    if remote_path in remote_files:

        print(
            f"[OK] Remote: {remote_path}"
        )

    else:

        print(
            f"[MISSING] Remote: {remote_path}"
        )

        missing_remote.append(
            asset_id
        )

print("\nVerification complete.")

if missing_remote:

    print(
        f"Missing: {len(missing_remote)}"
    )

else:

    print(
        "All selected HDRIs exist on "
        "Hugging Face."
    )

[OK] Remote: hdris/art_studio_2k.hdr
[OK] Remote: hdris/pav_studio_01_2k.hdr
[OK] Remote: hdris/wooden_studio_08_2k.hdr
[OK] Remote: hdris/wooden_studio_10_2k.hdr
[OK] Remote: hdris/monochrome_studio_02_2k.hdr
[OK] Remote: hdris/brown_photostudio_01_2k.hdr
[OK] Remote: hdris/turning_area_2k.hdr
[OK] Remote: hdris/teufelsberg_ground_1_2k.hdr
[OK] Remote: hdris/rural_evening_road_2k.hdr
[OK] Remote: hdris/spruit_sunrise_2k.hdr
[OK] Remote: hdris/ninomaru_teien_2k.hdr
[OK] Remote: hdris/bell_park_dawn_2k.hdr
[OK] Remote: hdris/san_giuseppe_bridge_2k.hdr
[OK] Remote: hdris/summer_stage_02_2k.hdr
[OK] Remote: hdris/zwartkops_curve_morning_2k.hdr
[OK] Remote: hdris/scythian_tombs_2k.hdr
[OK] Remote: hdris/golden_bay_2k.hdr
[OK] Remote: hdris/syferfontein_0d_clear_puresky_2k.hdr
[OK] Remote: hdris/kloofendal_43d_clear_2k.hdr
[OK] Remote: hdris/river_alcove_2k.hdr
[OK] Remote: hdris/immenstadter_horn_2k.hdr
[OK] Remote: hdris/cloud_layers_2k.hdr
[OK] Remote: hdris/autumn_hill_view_2k.hdr
[OK] 

In [40]:
# ============================================================
# CELL 29 — DELETE VERIFIED LOCAL HDRIs
# ============================================================

if missing_remote:

    raise RuntimeError(
        "Some HDRIs are missing from Hugging Face. "
        "Local files will NOT be deleted."
    )

deleted = 0

for asset_id in SELECTED_HDRIS:

    filename = f"{asset_id}_2k.hdr"

    local_path = (
        DOWNLOAD_DIR / filename
    )

    if local_path.exists():

        local_path.unlink()

        deleted += 1

        print(
            f"[DELETED] {filename}"
        )

print(
    f"\nDeleted {deleted} local HDRI files."
)

[DELETED] art_studio_2k.hdr
[DELETED] pav_studio_01_2k.hdr
[DELETED] wooden_studio_08_2k.hdr
[DELETED] wooden_studio_10_2k.hdr
[DELETED] monochrome_studio_02_2k.hdr
[DELETED] brown_photostudio_01_2k.hdr
[DELETED] turning_area_2k.hdr
[DELETED] teufelsberg_ground_1_2k.hdr
[DELETED] rural_evening_road_2k.hdr
[DELETED] spruit_sunrise_2k.hdr
[DELETED] ninomaru_teien_2k.hdr
[DELETED] bell_park_dawn_2k.hdr
[DELETED] san_giuseppe_bridge_2k.hdr
[DELETED] summer_stage_02_2k.hdr
[DELETED] zwartkops_curve_morning_2k.hdr
[DELETED] scythian_tombs_2k.hdr
[DELETED] golden_bay_2k.hdr
[DELETED] syferfontein_0d_clear_puresky_2k.hdr
[DELETED] kloofendal_43d_clear_2k.hdr
[DELETED] river_alcove_2k.hdr
[DELETED] immenstadter_horn_2k.hdr
[DELETED] cloud_layers_2k.hdr
[DELETED] autumn_hill_view_2k.hdr
[DELETED] lebombo_2k.hdr

Deleted 24 local HDRI files.


In [41]:
# ============================================================
# CELL 30 — FINAL STATE
# ============================================================

for asset_id in SELECTED_HDRIS:

    if asset_id in state["assets"]:

        state["assets"][asset_id][
            "local_deleted"
        ] = True

save_state()

print("=" * 70)
print("HDRI LIBRARY SETUP COMPLETE")
print("=" * 70)

print(
    f"Repository: "
    f"https://huggingface.co/datasets/{HF_REPO_ID}"
)

print(
    f"Selected HDRIs: {len(SELECTED_HDRIS)}"
)

print(
    f"Remote verified: "
    f"{len(SELECTED_HDRIS) - len(missing_remote)}"
)

print(
    f"Local HDRIs remaining: "
    f"{len(list(DOWNLOAD_DIR.glob('*.hdr')))}"
)

HDRI LIBRARY SETUP COMPLETE
Repository: https://huggingface.co/datasets/ndeda/neural-object-reconstruction-hdris
Selected HDRIs: 24
Remote verified: 24
Local HDRIs remaining: 0
